# EX_08 — Introducción a agentes (ejercicios)

**Notebook de referencia:** `notebook/08_Introduccion_Agentes.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Definir 2 herramientas

Escribe funciones Python puras `get_time_utc()` (puede ser fake) y `hash_text(s: str)` (usa `hashlib.sha256` en hex). Estas serán tus "tools".


In [2]:
import hashlib
from datetime import datetime, timezone

def get_time_utc() -> str:
    """
    Tool que devuelve la fecha y hora actual en UTC en formato ISO.
    """
    return datetime.now(timezone.utc).isoformat()


def hash_text(s: str) -> str:
    """
    Tool que recibe un texto y devuelve su hash SHA-256 en hexadecimal.
    """
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


In [3]:
print("UTC time:", get_time_utc())

text = "Hola mundo"
print("Texto:", text)
print("SHA-256:", hash_text(text))

UTC time: 2026-06-17T17:05:04.374743+00:00
Texto: Hola mundo
SHA-256: ca8f60b2cc7f05837d98b208b57fb6481553fc5f1219d59618fd025002a66f5c


## Actividad 2 — Cuándo usar tool

Para cada intención del usuario (`"What time is it?"`, `"Digest of hello"`), escribe en comentarios si el LLM debería llamar tool o responder directo.


In [4]:
# Actividad 2 — Cuándo usar tool

# Intención 1: "What time is it?"
# El LLM debería llamar a la tool get_time_utc(),
# porque la hora actual es información dinámica y no debe inventarla.
# Tool adecuada: get_time_utc()

# Intención 2: "Digest of hello"
# El LLM debería llamar a la tool hash_text("hello"),
# porque el usuario está pidiendo calcular el digest/hash de un texto.
# Tool adecuada: hash_text("hello")



intentions = [
    "What time is it?",
    "Digest of hello"
]

for intent in intentions:
    if "time" in intent.lower():
        print(f"{intent} -> CALL TOOL: get_time_utc()")
    elif "digest" in intent.lower() or "hash" in intent.lower():
        text = intent.replace("Digest of ", "")
        print(f"{intent} -> CALL TOOL: hash_text('{text}')")
    else:
        print(f"{intent} -> ANSWER DIRECTLY")

What time is it? -> CALL TOOL: get_time_utc()
Digest of hello -> CALL TOOL: hash_text('hello')


In [5]:
print(get_time_utc())
print(hash_text("hello"))

2026-06-17T17:05:51.505971+00:00
2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824


## Actividad 3 — Bucles

En español (celda markdown), explica el riesgo de **bucles infinitos** tool→modelo→tool y una mitigación (límite de pasos, detector de repetición).


### Actividad 3 — Riesgo de bucles en agentes con herramientas

En un sistema de agentes, existe el riesgo de que se produzca un bucle infinito entre el modelo y las herramientas. Esto puede ocurrir cuando el modelo decide llamar a una herramienta, recibe el resultado, pero no considera que la información sea suficiente y vuelve a llamar a la misma herramienta una y otra vez sin llegar a una respuesta final.

Por ejemplo, el flujo podría repetirse así:

`modelo → tool → modelo → tool → modelo → tool...`

Este comportamiento puede provocar consumo innecesario de recursos, aumento de coste, mayor latencia y bloqueo del sistema.

Una mitigación sencilla es establecer un límite máximo de pasos o iteraciones. Por ejemplo, si el agente ha usado herramientas más de 3 o 5 veces sin generar una respuesta final, el sistema debe detenerse y devolver una respuesta controlada indicando que no puede continuar con seguridad.

Otra mitigación es implementar un detector de repetición. Si el agente llama varias veces a la misma herramienta con los mismos argumentos y obtiene resultados similares, el sistema puede interpretar que está atrapado en un bucle y finalizar el proceso.

En resumen, los agentes deben tener mecanismos de control para evitar que la autonomía del modelo produzca ciclos infinitos. Los límites de pasos y la detección de repetición son dos estrategias básicas para hacer el sistema más robusto y seguro.
